### Gerando a tabela para processamento final

Para o processamento final é necessário realizar a junção de alguns arquivos para a extração das informações dos mesmos.

Diversas estratégias poderiam ter sido utilizadas aqui, como por exemplo a criação de um modelagem dimensional que possibilitasse melhor agragação dos dados e a realização das análises solicitadas. No entanto, por acreditar não ser o intuito principal do programa, optei por uma simples agregação com base na estrutura de dados Dataframe

In [ ]:
import pandas as pd
import os

base_path = '../data/Processed' 
files = {
    'deslocamento': os.path.join(base_path, 'Deslocamento_Enriquecido_com_UF_e_Codigos.csv'),
    'saneamento': os.path.join(base_path, 'Saneamento.csv'),
    'ensino': os.path.join(base_path, 'indicesEnsino.csv')
}

# --- Definição de Colunas Corretas ---
saneamento_cols_correct = [
    'sigla_uf', 'nome_uf', 'populacao_atendida_agua', 'populacao_atentida_esgoto',
    'volume_agua_produzido', 'volume_agua_tratada_eta', 'indice_coleta_esgoto',
    'indice_tratamento_esgoto', 'investimento_esgoto_estado',
    'investimento_recurso_proprio_estado', 'investimento_total_estado'
]

ensino_cols = [
    'sigla_uf', 'localizacao', 'rede', 'taxa_promocao_ef', 'taxa_promocao_ef_5_ano', 'taxa_promocao_ef_6_ano',
    'taxa_promocao_ef_7_ano', 'taxa_promocao_ef_8_ano', 'taxa_promocao_ef_9_ano', 'taxa_promocao_em',
    'taxa_promocao_em_1_ano', 'taxa_promocao_em_2_ano', 'taxa_promocao_em_3_ano', 'taxa_repetencia_ef',
    'taxa_repetencia_ef_5_ano', 'taxa_repetencia_ef_6_ano', 'taxa_repetencia_ef_7_ano', 'taxa_repetencia_ef_8_ano',
    'taxa_repetencia_ef_9_ano', 'taxa_repetencia_em', 'taxa_repetencia_em_1_ano', 'taxa_repetencia_em_2_ano',
    'taxa_repetencia_em_3_ano', 'taxa_evasao_ef', 'taxa_evasao_ef_5_ano', 'taxa_evasao_ef_6_ano',
    'taxa_evasao_ef_7_ano', 'taxa_evasao_ef_8_ano', 'taxa_evasao_ef_9_ano', 'taxa_evasao_em',
    'taxa_evasao_em_1_ano', 'taxa_evasao_em_2_ano', 'taxa_evasao_em_3_ano'
]


# 1. Carregar e Preparar Deslocamento
df_desloc_raw = pd.read_csv(files['deslocamento'], encoding='utf-8', header=None)
df_desloc_raw = df_desloc_raw.iloc[:, :-1]
df_desloc_raw.columns = ['Região Geográfica Intermediária', 'Grupo de idade', 'Pessoas', 'Tempo de Deslocamento', 'Nível de Instrução', 'Código Região Intermediária', 'Código UF', 'UF']
df_desloc_raw['Pessoas'] = pd.to_numeric(df_desloc_raw['Pessoas'], errors='coerce')
df_desloc_agg = df_desloc_raw[df_desloc_raw['Tempo de Deslocamento'] != 'Total'].copy()

tempo_map = {
    'Até cinco minutos': 2.5, 'Mais de cinco minutos até quinze minutos': 10, 'Mais de quinze minutos até meia hora': 22.5,
    'Mais de meia hora até uma hora': 45, 'Mais de uma hora até duas horas': 90, 'Mais de duas horas até quatro horas': 180,
    'Mais de quatro horas': 240, 'Sem informação': 0
}
df_desloc_agg['Tempo_Numerico'] = df_desloc_agg['Tempo de Deslocamento'].map(tempo_map)

df_desloc_uf = df_desloc_agg.groupby('UF').apply(
    lambda x: (x['Pessoas'] * x['Tempo_Numerico']).sum() / x['Pessoas'].sum()
).reset_index(name='Tempo_Deslocamento_Medio_Minutos')


# 2. Carregar e Preparar Saneamento
df_saneamento = pd.read_csv(files['saneamento'], encoding='utf-8', header=0)
df_saneamento.columns = saneamento_cols_correct
df_saneamento.rename(columns={'nome_uf': 'UF'}, inplace=True)
df_saneamento['Indice_Saneamento_Basico'] = (pd.to_numeric(df_saneamento['indice_coleta_esgoto'], errors='coerce') + pd.to_numeric(df_saneamento['indice_tratamento_esgoto'], errors='coerce')) / 2
df_saneamento_uf = df_saneamento[['UF', 'Indice_Saneamento_Basico']]


# 3. Carregar e Preparar Ensino
df_ensino = pd.read_csv(files['ensino'], encoding='utf-8', header=None)
df_ensino.columns = ensino_cols

# Filtrar apenas as linhas onde 'localizacao' e 'rede' são 'Total'
df_ensino_uf = df_ensino[(df_ensino['rede'] == 'Total') & (df_ensino['localizacao'] == 'Total')].copy()

df_ensino_uf['Taxa_Evasao_Media'] = (pd.to_numeric(df_ensino_uf['taxa_evasao_ef'], errors='coerce') + pd.to_numeric(df_ensino_uf['taxa_evasao_em'], errors='coerce')) / 2
df_ensino_uf['Taxa_Repetencia_Media'] = (pd.to_numeric(df_ensino_uf['taxa_repetencia_ef'], errors='coerce') + pd.to_numeric(df_ensino_uf['taxa_repetencia_em'], errors='coerce')) / 2
df_ensino_uf = df_ensino_uf[['sigla_uf', 'Taxa_Evasao_Media', 'Taxa_Repetencia_Media']].rename(columns={'sigla_uf': 'UF'})

uf_map = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas', 'BA': 'Bahia', 'CE': 'Ceará',
    'DF': 'Distrito Federal', 'ES': 'Espírito Santo', 'GO': 'Goiás', 'MA': 'Maranhão', 'MT': 'Mato Grosso',
    'MS': 'Mato Grosso do Sul', 'MG': 'Minas Gerais', 'PA': 'Pará', 'PB': 'Paraíba', 'PR': 'Paraná',
    'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima', 'SC': 'Santa Catarina',
    'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins'
}
df_ensino_uf['UF_Nome'] = df_ensino_uf['UF'].map(uf_map)
df_ensino_uf = df_ensino_uf.drop(columns=['UF']).rename(columns={'UF_Nome': 'UF'})


# --- Merge Final ---
df_final = pd.merge(df_desloc_uf, df_saneamento_uf, on='UF', how='inner')
df_final = pd.merge(df_final, df_ensino_uf, on='UF', how='inner')

# Arredondar os valores numéricos para duas casas decimais
cols_to_round = ['Tempo_Deslocamento_Medio_Minutos', 'Indice_Saneamento_Basico', 'Taxa_Evasao_Media', 'Taxa_Repetencia_Media']
df_final[cols_to_round] = df_final[cols_to_round].round(2)

df_final.to_csv('../data/Processed/dados_finais_analise.csv', index=False)

print("DataFrame Final (df_final) gerado com sucesso!")
print("\nPrimeiras 5 linhas do DataFrame Final:")
print(df_final.head())
print("\nO arquivo 'dados_finais_analise.csv' foi salvo no diretório atual.")


DataFrame Final (df_final) gerado com sucesso!

Primeiras 5 linhas do DataFrame Final:
         UF  Tempo_Deslocamento_Medio_Minutos  Indice_Saneamento_Basico  Taxa_Evasao_Media  Taxa_Repetencia_Media
0      Acre                             21.42                     59.86               5.20                   9.45
1   Alagoas                             18.76                     56.35               4.25                   2.30
2     Amapá                             24.06                     26.53               3.25                   8.05
3  Amazonas                             29.92                     19.70               1.90                   1.70
4     Bahia                             19.23                     56.67               4.25                   9.50

O arquivo 'dados_finais_analise.csv' foi salvo no diretório atual.


/tmp/ipykernel_83812/1088645801.py:49: RuntimeWarning: invalid value encountered in scalar divide
  lambda x: (x['Pessoas'] * x['Tempo_Numerico']).sum() / x['Pessoas'].sum()
/tmp/ipykernel_83812/1088645801.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_desloc_uf = df_desloc_agg.groupby('UF').apply(


### Geração das vizualizações e insights

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuração de exibição
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
# Definindo uma paleta de cores mais profissional
palette = sns.color_palette("viridis", 4)

caminho_salvar = '../data/vizualizations/'

# --- Carregar DataFrame Final ---
# Assumindo que o df_final já foi gerado e salvo com a estrutura correta e arredondamento
df_final = pd.read_csv('../data/Processed/dados_finais_analise.csv')
cols_to_round = ['Tempo_Deslocamento_Medio_Minutos', 'Indice_Saneamento_Basico', 'Taxa_Evasao_Media', 'Taxa_Repetencia_Media']


# 1. Impacto do Deslocamento na Evasão (Scatter Plot Simples)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Tempo_Deslocamento_Medio_Minutos', y='Taxa_Evasao_Media', data=df_final, s=100, alpha=0.8, color=palette[0])
for i in range(df_final.shape[0]):
    plt.text(df_final['Tempo_Deslocamento_Medio_Minutos'].iloc[i] + 0.2, df_final['Taxa_Evasao_Media'].iloc[i], df_final['UF'].iloc[i], fontsize=9, alpha=0.8)
plt.title('Insight 1: Relação entre Tempo de Deslocamento e Evasão Escolar por UF', fontsize=14)
plt.xlabel('Tempo de Deslocamento Médio (minutos)', fontsize=12)
plt.ylabel('Taxa de Evasão Média (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(caminho_salvar + 'Insight_1_Deslocamento_vs_Evasao_Simples.png', bbox_inches='tight')
plt.close()


# 2. Impacto do Saneamento na Evasão (Scatter Plot Simples)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Indice_Saneamento_Basico', y='Taxa_Evasao_Media', data=df_final, s=100, alpha=0.8, color=palette[1])
for i in range(df_final.shape[0]):
    plt.text(df_final['Indice_Saneamento_Basico'].iloc[i] + 0.5, df_final['Taxa_Evasao_Media'].iloc[i], df_final['UF'].iloc[i], fontsize=9, alpha=0.8)
plt.title('Insight 2: Relação entre Saneamento Básico e Evasão Escolar por UF', fontsize=14)
plt.xlabel('Índice de Saneamento Básico (%)', fontsize=12)
plt.ylabel('Taxa de Evasão Média (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(caminho_salvar + 'Insight_2_Saneamento_vs_Evasao_Simples.png', bbox_inches='tight')
plt.close()


# 3. Impacto do Saneamento na Repetência (Scatter Plot Simples)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Indice_Saneamento_Basico', y='Taxa_Repetencia_Media', data=df_final, s=100, alpha=0.8, color=palette[2])
for i in range(df_final.shape[0]):
    plt.text(df_final['Indice_Saneamento_Basico'].iloc[i] + 0.5, df_final['Taxa_Repetencia_Media'].iloc[i], df_final['UF'].iloc[i], fontsize=9, alpha=0.8)
plt.title('Insight 3: Relação entre Saneamento Básico e Taxa de Repetência por UF', fontsize=14)
plt.xlabel('Índice de Saneamento Básico (%)', fontsize=12)
plt.ylabel('Taxa de Repetência Média (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(caminho_salvar + 'Insight_3_Saneamento_vs_Repetencia_Simples.png', bbox_inches='tight')
plt.close()

# 4. Heatmap de Correlação (Melhor Paleta de Cores e Formato)
plt.figure(figsize=(9, 8))
corr_matrix = df_final[cols_to_round].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f",
            linewidths=1, linecolor='white', cbar_kws={'label': 'Coeficiente de Correlação'})
plt.title('Insight 4: Matriz de Correlação entre Variáveis Chave', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.savefig(caminho_salvar + 'Insight_4_Heatmap_Correlacao_Refinado.png', bbox_inches='tight')
plt.close()

# 5. Gráfico de Barras Combinado: Top 5 e Bottom 5 em Taxa de Evasão (Melhor Dual-Axis)
df_top_evasao = df_final.sort_values(by='Taxa_Evasao_Media', ascending=False).head(5)
df_bottom_evasao = df_final.sort_values(by='Taxa_Evasao_Media', ascending=True).head(5)
df_comparacao = pd.concat([df_top_evasao, df_bottom_evasao])
df_comparacao = df_comparacao.sort_values(by='Taxa_Evasao_Media', ascending=False)

fig, ax1 = plt.subplots(figsize=(14, 7))
sns.barplot(x='UF', y='Taxa_Evasao_Media', data=df_comparacao, ax=ax1, color=palette[0], label='Taxa de Evasão Média (%)')
ax1.set_ylabel('Taxa de Evasão Média (%)', color=palette[0], fontsize=12)
ax1.tick_params(axis='y', labelcolor=palette[0])
ax1.set_xlabel('Unidade Federativa (UF)', fontsize=12)
ax1.set_title('Insight 5: Comparação de Taxa de Evasão Média e Tempo de Deslocamento (Top/Bottom 5)', fontsize=14)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(False)

ax2 = ax1.twinx()
sns.lineplot(x='UF', y='Tempo_Deslocamento_Medio_Minutos', data=df_comparacao, ax=ax2, color=palette[3], marker='o', linewidth=2, markersize=8, label='Tempo de Deslocamento Médio (min)')
ax2.set_ylabel('Tempo de Deslocamento Médio (min)', color=palette[3], fontsize=12)
ax2.tick_params(axis='y', labelcolor=palette[3])
ax2.grid(True, linestyle='--', alpha=0.6)

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper right')

plt.savefig(caminho_salvar + 'Insight_5_Comparacao_Evasao_Deslocamento_Refinado.png', bbox_inches='tight')
plt.close()

# 6. Boxplot para Distribuição das Métricas (Melhor Rótulo)
df_melt = df_final[cols_to_round].melt(var_name='Métrica', value_name='Valor')
plt.figure(figsize=(12, 6))
sns.boxplot(x='Métrica', y='Valor', data=df_melt, palette='Pastel1')
plt.title('Insight 6: Distribuição das Métricas por UF (Boxplot)', fontsize=14)
plt.xlabel('Métrica', fontsize=12)
plt.ylabel('Valor', fontsize=12)
plt.xticks(rotation=15, ha='right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(caminho_salvar + 'Insight_6_Boxplot_Distribuicao_Refinado.png', bbox_inches='tight')
plt.close()


# 7. Gráfico de Barras: Tempo de Deslocamento Médio por UF (Ordenado)
df_sorted_desloc = df_final.sort_values(by='Tempo_Deslocamento_Medio_Minutos', ascending=False)
plt.figure(figsize=(14, 7))
sns.barplot(x='UF', y='Tempo_Deslocamento_Medio_Minutos', data=df_sorted_desloc, palette='magma')
plt.title('Insight 7: Tempo de Deslocamento Médio por UF (Ordenado)', fontsize=14)
plt.xlabel('Unidade Federativa (UF)', fontsize=12)
plt.ylabel('Tempo de Deslocamento Médio (minutos)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.savefig(caminho_salvar + 'Insight_7_Tempo_Deslocamento_por_UF.png', bbox_inches='tight')
plt.close()

# 8. Gráfico de Barras: Índice de Saneamento Básico por UF (Ordenado)
df_sorted_saneamento = df_final.sort_values(by='Indice_Saneamento_Basico', ascending=False)
plt.figure(figsize=(14, 7))
sns.barplot(x='UF', y='Indice_Saneamento_Basico', data=df_sorted_saneamento, palette='crest')
plt.title('Insight 8: Índice de Saneamento Básico por UF (Ordenado)', fontsize=14)
plt.xlabel('Unidade Federativa (UF)', fontsize=12)
plt.ylabel('Índice de Saneamento Básico (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.savefig(caminho_salvar + 'Insight_8_Saneamento_por_UF.png', bbox_inches='tight')
plt.close()

# 9. Gráfico de Barras: Taxa de Evasão Média por UF (Ordenado)
df_sorted_evasao = df_final.sort_values(by='Taxa_Evasao_Media', ascending=False)
plt.figure(figsize=(14, 7))
sns.barplot(x='UF', y='Taxa_Evasao_Media', data=df_sorted_evasao, palette='Reds_r')
plt.title('Insight 9: Taxa de Evasão Média por UF (Ordenado)', fontsize=14)
plt.xlabel('Unidade Federativa (UF)', fontsize=12)
plt.ylabel('Taxa de Evasão Média (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.savefig(caminho_salvar + 'Insight_9_Evasao_por_UF.png', bbox_inches='tight')
plt.close()


/tmp/ipykernel_83812/256870174.py:104: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='Métrica', y='Valor', data=df_melt, palette='Pastel1')
/tmp/ipykernel_83812/256870174.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='UF', y='Tempo_Deslocamento_Medio_Minutos', data=df_sorted_desloc, palette='magma')
/tmp/ipykernel_83812/256870174.py:130: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='UF', y='Indice_Saneamento_Basico', data=df_sorted_saneamento, palette='crest')
/tmp/ipykernel_83812/256870174.py:141: FutureWarning: 

Passing `pa